# Model Architectures

In [1]:
import pandas as pd
import torch

from src.models import build_model, count_parameters

## Baseline CNN

Three convolution blocks (32 -> 64 -> 128), each consisting of Conv2D -> BatchNorm -> ReLU -> MaxPool -> Dropout2d.

Global average pooling replaces a flatten layer to keep the number of parameters low.

In [2]:
baseline = build_model("baseline", pretrained=False)
print(baseline)
print()

total, trainable = count_parameters(baseline)

print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

dummy = torch.randn(1, 1, 224, 224)
output = baseline(dummy)
print(f"Output shape: {output.shape}")

BaselineCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Dropout2d(p=0.3, inplace=False)
    )
    (1): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Dropout2d(p=0.3, inplace=False)
    )
    (2): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=

## Deeper CNN

Regularised CNN with five convolution blocks (32 -> 64 -> 128 -> 256 -> 512), to test if a deeper network can improve performance, while keeping the same head structure as the Baseline CNN.

In [3]:
deeper = build_model("deeper", pretrained=False)
total, trainable = count_parameters(deeper)

print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

output = deeper(dummy)
print(f"Output shape: {output.shape}")

Total parameters: 1,700,577
Trainable parameters: 1,700,577
Output shape: torch.Size([1, 1])


## Transfer Learning Models

Pretrained ImageNet backbones with a custom classification head: Dropout -> Linear -> ReLU -> Dropout -> Linear(1).

In [4]:
tl_models = ["vgg16", "resnet50"]

for model in tl_models:
    # Avoid downloading pretrained weights in the notebook
    tl_model = build_model(model, pretrained=False)
    total, trainable = count_parameters(tl_model)

    print(f"Model: {model}")
    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters: {trainable:,}")

    output = tl_model(dummy)
    print(f"Output shape: {output.shape}\n")

Model: vgg16
Total parameters: 135,309,633
Trainable parameters: 135,309,633
Output shape: torch.Size([1, 1])

Model: resnet50
Total parameters: 24,032,833
Trainable parameters: 24,032,833
Output shape: torch.Size([1, 1])



## Parameter Comparison

In [5]:
models = ["baseline", "deeper", "vgg16", "resnet50"]

rows = []
for name in models:
    model = build_model(name, pretrained=False)
    total, trainable = count_parameters(model)
    rows.append({"Model": name, "Total Parameters": f"{total:,}", "Trainable Parameters": f"{trainable:,}"})

display(pd.DataFrame(rows))

,Model,Total Parameters,Trainable Parameters
0,baseline,"126,177","126,177"
1,deeper,"1,700,577","1,700,577"
2,vgg16,"135,309,633","135,309,633"
3,resnet50,"24,032,833","24,032,833"
